In [1]:
!pip install transformers accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00


In [2]:
!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.5/627.5 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/1

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!ls /content/drive/MyDrive/

 adapters  'Colab Notebooks'   model-fp16   tinyllama-adapters


In [7]:
MODEL_PATH = "/content/drive/MyDrive/llm-project/model-int5"

In [8]:
import time
import torch

def benchmark(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    torch.cuda.empty_cache()
    start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            use_cache=True
        )

    end = time.time()

    tokens = output.shape[1] - inputs.input_ids.shape[1]
    latency = end - start
    tps = tokens / latency

    vram = torch.cuda.memory_allocated() / 1e9

    return tps, latency, vram

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "/content/drive/MyDrive/tinyllama-adapters"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, adapter_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [14]:
model = model.merge_and_unload()

In [15]:
save_path = "/content/drive/MyDrive/model-fp16"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/model-fp16/tokenizer_config.json',
 '/content/drive/MyDrive/model-fp16/special_tokens_map.json',
 '/content/drive/MyDrive/model-fp16/chat_template.jinja',
 '/content/drive/MyDrive/model-fp16/tokenizer.model',
 '/content/drive/MyDrive/model-fp16/added_tokens.json',
 '/content/drive/MyDrive/model-fp16/tokenizer.json')

In [16]:
!ls /content/drive/MyDrive/model-fp16

chat_template.jinja	model.safetensors	 tokenizer.json
config.json		special_tokens_map.json  tokenizer.model
generation_config.json	tokenizer_config.json


In [17]:
MODEL_PATH = "/content/drive/MyDrive/model-fp16"

In [18]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    save_path,
    device_map="auto",
    quantization_config=bnb_config
)

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_PATH = "/content/drive/MyDrive/model-fp16"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.float16
)

In [20]:
import time

def benchmark(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    torch.cuda.empty_cache()

    start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            use_cache=True
        )

    end = time.time()

    tokens = output.shape[1] - inputs.input_ids.shape[1]
    latency = end - start
    tps = tokens / latency
    vram = torch.cuda.memory_allocated() / 1e9

    return tps, latency, vram

In [21]:
prompts = [
    "Explain quantization in simple terms",
    "What is LoRA in AI?",
    "Write Python code for Fibonacci"
]

for p in prompts:
    tps, latency, vram = benchmark(model, tokenizer, p)

    print(f"\nPrompt: {p}")
    print(f"Tokens/sec: {tps:.2f}")
    print(f"Latency: {latency:.2f}s")
    print(f"VRAM: {vram:.2f} GB")


Prompt: Explain quantization in simple terms
Tokens/sec: 1.64
Latency: 60.83s
VRAM: 0.00 GB

Prompt: What is LoRA in AI?
Tokens/sec: 2.59
Latency: 38.67s
VRAM: 0.00 GB

Prompt: Write Python code for Fibonacci
Tokens/sec: 2.68
Latency: 37.25s
VRAM: 0.00 GB


In [1]:
!nvidia-smi

Fri Apr 10 10:35:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_PATH = "/content/drive/MyDrive/model-fp16"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",   # 👈 important
    torch_dtype=torch.float16
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [2]:
import time
import torch

def benchmark(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    torch.cuda.empty_cache()

    start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            use_cache=True
        )

    end = time.time()

    tokens = output.shape[1] - inputs.input_ids.shape[1]
    latency = end - start
    tps = tokens / latency

    vram = torch.cuda.memory_reserved() / 1e9

    return tps, latency, vram

In [6]:
prompts = [
    "Explain quantization in simple terms",
    "What is LoRA in AI?",
    "Write Python code for Fibonacci"
]

In [7]:
for p in prompts:
    tps, latency, vram = benchmark(model, tokenizer, p)

    print(f"\nPrompt: {p}")
    print(f"Tokens/sec: {tps:.2f}")
    print(f"Latency: {latency:.2f}s")
    print(f"VRAM: {vram:.2f} GB")

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: Explain quantization in simple terms
Tokens/sec: 12.25
Latency: 8.17s
VRAM: 2.05 GB


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: What is LoRA in AI?
Tokens/sec: 10.80
Latency: 9.26s
VRAM: 2.05 GB

Prompt: Write Python code for Fibonacci
Tokens/sec: 10.23
Latency: 9.78s
VRAM: 2.05 GB


In [26]:
results = {
    "FP16": {"tps": 30, "latency": 3, "vram": 2.25},
    "INT8": {"tps": 9, "latency": 10, "vram": 1.37},
    "INT4": {"tps": 11, "latency": 9, "vram": 2.05},
}

In [5]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [17]:
!pip install -U bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_PATH = "/content/drive/MyDrive/model-fp16"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [27]:
results = {
    "FP16": {"tps": 30, "latency": 3, "vram": 2.25},
    "INT8": {"tps": 9, "latency": 10, "vram": 1.37},
    "INT4": {"tps": 11, "latency": 9, "vram": 2.05},
}

In [11]:
import json

with open("results.json", "w") as f:
    json.dump(results, f)

In [12]:
import pandas as pd

data = [
    ["FP16", 30, 3, 2.25],
    ["INT8", 9, 10, 1.37],
    ["INT4", 11, 9, 2.05],
]

df = pd.DataFrame(data, columns=["Mode", "Tokens/sec", "Latency", "VRAM"])
df.to_csv("results.csv", index=False)

In [13]:
from google.colab import files
files.download("results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import time
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ========================
# CONFIG
# ========================
MODEL_PATH = "/content/drive/MyDrive/model-fp16"

prompts = [
    "Explain quantization in simple terms",
    "What is LoRA in AI?",
    "Write Python code for Fibonacci"
]

# ========================
# LOAD MODEL (CHANGE MODE HERE)
# ========================

def load_model(mode="fp16"):
    if mode == "fp16":
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.float16,
            device_map="auto"
        )

    elif mode == "int8":
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            quantization_config=bnb_config,
            device_map="auto"
        )

    elif mode == "int4":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            quantization_config=bnb_config,
            device_map="auto"
        )

    else:
        raise ValueError("Invalid mode")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    return model, tokenizer


# ========================
# BENCHMARK FUNCTION
# ========================

def benchmark(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    torch.cuda.empty_cache()

    start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            use_cache=True
        )

    end = time.time()

    tokens = output.shape[1] - inputs.input_ids.shape[1]
    latency = end - start
    tps = tokens / latency

    vram = torch.cuda.memory_reserved() / 1e9

    return tps, latency, vram


# ========================
# BATCH INFERENCE
# ========================

def batch_inference(model, tokenizer, prompts):
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

    start = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100
        )

    end = time.time()

    latency = end - start
    return latency


# ========================
# STREAMING (SIMULATION)
# ========================

def stream_output(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    print("\nStreaming Output:\n")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=50
        )

    text = tokenizer.decode(output[0])

    for word in text.split():
        print(word, end=" ", flush=True)
        time.sleep(0.05)


# ========================
# RUN BENCHMARK
# ========================

def run(mode):
    model, tokenizer = load_model(mode)

    results = []

    print(f"\nRunning mode: {mode.upper()}\n")

    for p in prompts:
        tps, latency, vram = benchmark(model, tokenizer, p)

        print(f"\nPrompt: {p}")
        print(f"Tokens/sec: {tps:.2f}")
        print(f"Latency: {latency:.2f}s")
        print(f"VRAM: {vram:.2f} GB")

        results.append([mode, p, tps, latency, vram])

    return results


# ========================
# MAIN
# ========================

if __name__ == "__main__":
    all_results = []

    for mode in ["fp16", "int8", "int4"]:
        res = run(mode)
        all_results.extend(res)

    # Save CSV
    df = pd.DataFrame(all_results, columns=["Mode", "Prompt", "TPS", "Latency", "VRAM"])
    df.to_csv("results.csv", index=False)

    print("\n✅ Results saved to results.csv")

    # Batch test
    model, tokenizer = load_model("fp16")
    batch_latency = batch_inference(model, tokenizer, prompts)
    print(f"\nBatch Latency: {batch_latency:.2f}s")

    # Streaming demo
    stream_output(model, tokenizer, prompts[0])

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Running mode: FP16



Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: What is LoRA in AI?
Tokens/sec: 25.21
Latency: 3.97s
VRAM: 2.60 GB

Prompt: Write Python code for Fibonacci
Tokens/sec: 23.18
Latency: 4.31s
VRAM: 2.60 GB

✅ Results saved to results.csv


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Batch Latency: 2.85s

Streaming Output:

<s> Explain quantization in simple terms. Answer: Quantization is a process of converting a digital value into a more manageable range of values. In computer science, it is used to reduce the size of data transmitted over a network or stored in a memory. 

In [16]:
!python test_inference.py

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 201/201 [00:03<00:00, 61.43it/s]

Running mode: FP16

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

Prompt: Explain quantization in simple terms
Tokens/sec: 29.42
Latency: 3.40s
VRAM: 2.25 GB
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

Prompt: What is LoRA in AI?
Tokens/sec: 34.97
Latency: 2.86s
VRAM: 2.25 GB
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/tra

In [18]:
!mkdir -p benchmarks
!mv results.csv benchmarks/

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import os
os.makedirs("/content/drive/MyDrive/benchmarks", exist_ok=True)

In [22]:
df.to_csv("/content/drive/MyDrive/benchmarks/results.csv", index=False)

In [23]:
tokenizer.padding_side = "left"